# SIGNAL — Data Cleaning

**Notebook:** `notebooks/02_data_cleaning/01_data_cleaning.ipynb`  
**Purpose:** Clean and prepare the SAKERNAS 2023 microdata for income-class classification and XAI analysis.

### Decisions applied
- Binary encoding for gender, disability, digital_use, social_participation, agri_employment
- Ordinal encoding for disability_severity
- Keep disability_type as binary indicator flags (preserves story)
- Create `digital_status` (0 = no digital, 1 = digital but no internet, 2 = uses internet)
- Coarsen province → region (Java, Sumatra, Kalimantan, Sulawesi, Bali_Nusa, Eastern)
- Create income_class (tertiles)
- Drop derived / redundant columns
- Prefer weekly_workhours over monthly_workhours

## 1. Setup & Load Data

In [ ]:
# Install if running in Colab
# !pip install -q huggingface_hub pandas scikit-learn

import pandas as pd
import numpy as np
from pathlib import Path

# For Colab / Hugging Face download
try:
    from huggingface_hub import hf_hub_download
    HF_AVAILABLE = True
except ImportError:
    HF_AVAILABLE = False

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 120)

In [ ]:
# -------------------------------------------------
# Load data
# -------------------------------------------------
# Option A: From Hugging Face (recommended)
if HF_AVAILABLE:
    file_path = hf_hub_download(
        repo_id="Sakhiur/signal",
        filename="data_in_brief.csv",
        repo_type="dataset"          # change to "model" if needed
    )
    df = pd.read_csv(file_path)
else:
    # Option B: Local path (adjust if needed)
    df = pd.read_csv("../../data/raw/data_in_brief.csv")

print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())
df.head()

## 2. Quick Inspection

In [ ]:
print("Missing values:\n")
print(df.isnull().sum().sort_values(ascending=False))

print("\n\nDtypes:\n")
print(df.dtypes)

print("\n\nUnique values (categorical candidates):")
for col in ["gender", "disability", "disability_severity", "disability_type",
            "digital_use", "internet_use", "social_participation",
            "agri_employment", "digital_index", "social_participation_index"]:
    if col in df.columns:
        print(f"\n{col}:")
        print(df[col].value_counts(dropna=False))

## 3. Cleaning Pipeline

In [ ]:
df_clean = df.copy()
print("Starting shape:", df_clean.shape)

### 3.1 Binary Encoding

In [ ]:
# ---- Gender ----
# Adjust the labels if your data uses different strings
df_clean["gender_male"] = (df_clean["gender"].str.lower().str.strip() == "male").astype(int)

# ---- Disability (binary) ----
df_clean["disabled"] = (df_clean["disability"].str.lower().str.strip() == "disabled").astype(int)

# ---- Digital use (temporary, will be replaced by digital_status) ----
df_clean["is_digital_user"] = (
    df_clean["digital_use"].str.lower().str.contains("digital", na=False)
).astype(int)

# ---- Social participation ----
df_clean["social_participant"] = (
    df_clean["social_participation"].str.lower().str.contains("participant", na=False)
).astype(int)

# ---- Agricultural employment ----
df_clean["agri_worker"] = (
    df_clean["agri_employment"].str.lower().str.contains("agriculture", na=False)
).astype(int)

print("Binary columns created: gender_male, disabled, is_digital_user, social_participant, agri_worker")
df_clean[["gender", "gender_male", "disability", "disabled"]].head()

### 3.2 Disability Severity (Ordinal)

In [ ]:
severity_map = {
    "non-disabled": 0,
    "non disabled": 0,
    "mild disability": 1,
    "mild": 1,
    "severe disability": 2,
    "severe": 2,
    "total disability": 3,
    "total": 3,
}

df_clean["disability_severity_ord"] = (
    df_clean["disability_severity"]
    .str.lower()
    .str.strip()
    .map(severity_map)
)

print("disability_severity_ord value counts:")
print(df_clean["disability_severity_ord"].value_counts(dropna=False))

### 3.3 Disability Type → Binary Flags (keep the story)

In [ ]:
# Inspect actual values first
print("Unique disability_type values:")
print(df_clean["disability_type"].value_counts(dropna=False))

In [ ]:
# Create binary flags. Non-disabled people get 0 for all types.
# Adjust the list below to match the exact strings in your data.

type_mapping = {
    "vision": "disability_vision",
    "mental": "disability_mental",
    "walking": "disability_walking",
    "hearing": "disability_hearing",
    "hand/finger disability": "disability_hand_finger",
    "hand/finger": "disability_hand_finger",
    "hand": "disability_hand_finger",
    "finger": "disability_hand_finger",
}

# Initialise all flags to 0
for new_col in set(type_mapping.values()):
    df_clean[new_col] = 0

# Set flag = 1 where the type matches
for raw_value, new_col in type_mapping.items():
    mask = df_clean["disability_type"].str.lower().str.strip() == raw_value
    df_clean.loc[mask, new_col] = 1

print("\nDisability type flags created:")
flag_cols = [c for c in df_clean.columns if c.startswith("disability_") and c not in ["disability", "disability_severity", "disability_severity_ord", "disability_type"]]
print(flag_cols)
print(df_clean[flag_cols].sum())

### 3.4 Digital Status (0 / 1 / 2)

In [ ]:
def create_digital_status(row):
    """
    0 = Non-digital
    1 = Digital user but does not use internet
    2 = Uses internet
    """
    digital = str(row.get("digital_use", "")).lower()
    internet = str(row.get("internet_use", "")).lower()

    if "non-digital" in digital or "non digital" in digital or digital == "0":
        return 0
    # digital user
    if "do not use" in internet or "not use" in internet or internet in ["0", "no"]:
        return 1
    return 2

df_clean["digital_status"] = df_clean.apply(create_digital_status, axis=1)

print("digital_status distribution:")
print(df_clean["digital_status"].value_counts().sort_index())
print("\nCross-tab digital_use × internet_use (for verification):")
print(pd.crosstab(df_clean["digital_use"], df_clean["internet_use"], margins=True))

### 3.5 Province → Region

In [ ]:
# Standard BPS province codes → broad region
province_to_region = {
    # Sumatra
    11: "Sumatra", 12: "Sumatra", 13: "Sumatra", 14: "Sumatra", 15: "Sumatra",
    16: "Sumatra", 17: "Sumatra", 18: "Sumatra", 19: "Sumatra", 21: "Sumatra",
    # Java
    31: "Java", 32: "Java", 33: "Java", 34: "Java", 35: "Java", 36: "Java",
    # Bali & Nusa Tenggara
    51: "Bali_Nusa", 52: "Bali_Nusa", 53: "Bali_Nusa",
    # Kalimantan
    61: "Kalimantan", 62: "Kalimantan", 63: "Kalimantan", 64: "Kalimantan", 65: "Kalimantan",
    # Sulawesi
    71: "Sulawesi", 72: "Sulawesi", 73: "Sulawesi", 74: "Sulawesi", 75: "Sulawesi", 76: "Sulawesi",
    # Eastern Indonesia (Maluku + Papua)
    81: "Eastern", 82: "Eastern",
    91: "Eastern", 92: "Eastern", 94: "Eastern", 95: "Eastern", 96: "Eastern", 97: "Eastern",
}

df_clean["region"] = df_clean["province"].map(province_to_region)

print("Region distribution:")
print(df_clean["region"].value_counts(dropna=False))

unmapped = df_clean.loc[df_clean["region"].isna(), "province"].unique()
if len(unmapped) > 0:
    print("\n⚠️ Unmapped province codes (please check):", unmapped)
else:
    print("\n✅ All provinces mapped successfully.")

### 3.6 Income Class (Target)

In [ ]:
# Tertiles based on monthly_income
df_clean["income_class"] = pd.qcut(
    df_clean["monthly_income"],
    q=3,
    labels=["Low", "Medium", "High"]
)

print("Income class distribution:")
print(df_clean["income_class"].value_counts().sort_index())
print("\nIncome statistics by class:")
print(df_clean.groupby("income_class", observed=True)["monthly_income"].describe().round(0))

### 3.7 Drop Columns We No Longer Need

In [ ]:
cols_to_drop = [
    # Original categoricals that we encoded
    "gender",
    "disability",
    "disability_severity",
    "disability_type",
    "digital_use",
    "internet_use",
    "social_participation",
    "agri_employment",
    "province",                    # replaced by region

    # Derived / redundant
    "agri_productivity",
    "agri_productivity_trimmed",
    "log_monthly_income",          # used only to understand distribution
    "monthly_workhours",           # keep weekly_workhours instead
    "monthly_income",              # target is now income_class

    # Temporary helper
    "is_digital_user",
]

# Only drop columns that actually exist
cols_to_drop = [c for c in cols_to_drop if c in df_clean.columns]
df_clean = df_clean.drop(columns=cols_to_drop)

print("Dropped columns:", cols_to_drop)
print("\nRemaining columns:")
print(df_clean.columns.tolist())
print("\nFinal shape:", df_clean.shape)

## 4. Final Checks

In [ ]:
print("Missing values after cleaning:")
print(df_clean.isnull().sum().sort_values(ascending=False))

print("\n\nData types:")
print(df_clean.dtypes)

print("\n\nSample of cleaned data:")
df_clean.head()

In [ ]:
# Optional: one-hot encode region if you prefer (tree models can handle string categories)
# Uncomment if needed:
# df_clean = pd.get_dummies(df_clean, columns=["region"], prefix="region", drop_first=False)

## 5. Save Cleaned Dataset

In [ ]:
output_dir = Path("../../data/processed")
output_dir.mkdir(parents=True, exist_ok=True)

output_path = output_dir / "signal_cleaned.csv"
df_clean.to_csv(output_path, index=False)

print(f"✅ Cleaned dataset saved to: {output_path}")
print(f"   Shape: {df_clean.shape}")
print(f"   Columns: {df_clean.columns.tolist()}")

## 6. Feature List for Modeling (Quick Reference)

**Target:** `income_class` (Low / Medium / High)

**Features (recommended):**
- `age`
- `education_years`
- `gender_male`
- `disabled`
- `disability_severity_ord`
- `disability_vision`, `disability_mental`, `disability_walking`, `disability_hearing`, `disability_hand_finger`
- `weekly_workhours`
- `digital_status` (0/1/2)
- `digital_index`
- `social_participant`
- `social_participation_index`
- `agri_worker`
- `region` (or one-hot versions)

You are now ready for EDA refinement or modeling.